In [ ]:
import pandas as pd
import glob
import re

def clean_text(text):
    """Fungsi untuk membersihkan teks ulasan dari URL, enter, dan spasi berlebih."""
    text = str(text)
    # Menghapus tautan / URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Mengganti karakter baris baru (\n) dengan spasi
    text = re.sub(r'\n+', ' ', text)
    # Menghapus spasi ganda atau berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def main():
    print("Memulai proses penggabungan dan pembersihan data...")

    # 1. Mendeteksi dan mengurutkan ke-12 file secara otomatis
    file_pattern = 'dataset_Google-Maps-Reviews-Scraper_*.csv'
    file_list = sorted(glob.glob(file_pattern))

    if not file_list:
        print("Error: Tidak ada file CSV yang ditemukan dengan pola tersebut di folder ini.")
        return

    print(f"\nDitemukan {len(file_list)} file:")
    for f in file_list:
        print(f" - {f}")

    # 2. Membaca dan menggabungkan semua file menjadi satu DataFrame
    df_list = [pd.read_csv(file) for file in file_list]
    df_combined = pd.concat(df_list, ignore_index=True)
    print(f"\nTotal baris awal dari semua file: {len(df_combined)}")

    # 3. Memilih kolom yang relevan (termasuk metadata untuk kebutuhan Backend/Database)
    columns_to_keep = [
        'title',
        'name',
        'text',
        'stars',
        'location/lat',
        'location/lng',
        'placeId',
        'reviewId',
        'address',
        'publishedAtDate'
    ]

    # Memastikan hanya mengambil kolom yang benar-benar ada di dataset untuk mencegah error
    existing_columns = [col for col in columns_to_keep if col in df_combined.columns]
    df_clean = df_combined[existing_columns].copy()

    # 4. Menghapus data kosong dan duplikat berdasarkan teks ulasan
    df_clean = df_clean.dropna(subset=['text'])
    df_clean = df_clean.drop_duplicates(subset=['text'])

    # 5. Menerapkan pembersihan teks menggunakan fungsi NLP
    df_clean['cleaned_text'] = df_clean['text'].apply(clean_text)

    # Menghapus baris yang mungkin menjadi string kosong ("") setelah dibersihkan
    df_clean = df_clean[df_clean['cleaned_text'] != ""]

    print(f"Total baris setelah dibersihkan (tanpa duplikat & nilai kosong): {len(df_clean)}")

    # 6. Menyimpan hasil ke dalam dua file CSV terpisah

    # File A: Master Dataset (Berisi identitas lengkap untuk Database & integrasi UI)
    df_clean.to_csv('Master_Dataset_Reviews_Cleaned.csv', index=False)
    print("\n[V] Master dataset berhasil disimpan sebagai 'Master_Dataset_Reviews_Cleaned.csv'")

    # File B: Training Dataset (Hanya berisi fitur teks dan label bintang untuk training model AI)
    df_training = df_clean[['cleaned_text', 'stars']]
    df_training.to_csv('Training_Dataset.csv', index=False)
    print("[V] Training dataset berhasil disimpan sebagai 'Training_Dataset.csv'")

    print("\nProses selesai! Data sudah siap digunakan untuk pemodelan sentimen dan injeksi ke database.")

if __name__ == "__main__":
    main()

Memulai proses penggabungan dan pembersihan data...

Ditemukan 12 file:
 - dataset_Google-Maps-Reviews-Scraper_001.csv
 - dataset_Google-Maps-Reviews-Scraper_002.csv
 - dataset_Google-Maps-Reviews-Scraper_003.csv
 - dataset_Google-Maps-Reviews-Scraper_004.csv
 - dataset_Google-Maps-Reviews-Scraper_005.csv
 - dataset_Google-Maps-Reviews-Scraper_006.csv
 - dataset_Google-Maps-Reviews-Scraper_007.csv
 - dataset_Google-Maps-Reviews-Scraper_008.csv
 - dataset_Google-Maps-Reviews-Scraper_009.csv
 - dataset_Google-Maps-Reviews-Scraper_010.csv
 - dataset_Google-Maps-Reviews-Scraper_011.csv
 - dataset_Google-Maps-Reviews-Scraper_012.csv

Total baris awal dari semua file: 20597
Total baris setelah dibersihkan (tanpa duplikat & nilai kosong): 12649

[V] Master dataset berhasil disimpan sebagai 'Master_Dataset_Reviews_Cleaned.csv'
[V] Training dataset berhasil disimpan sebagai 'Training_Dataset.csv'

Proses selesai! Data sudah siap digunakan untuk pemodelan sentimen dan injeksi ke database.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import joblib
from sklearn.metrics import classification_report, confusion_matrix

# 1. Memuat Dataset Training
df = pd.read_csv('Training_Dataset.csv')

# Asumsikan kita ubah 'stars' menjadi kelas teks:
# 1-2 = Negatif, 3 = Netral, 4-5 = Positif
def convert_stars_to_sentiment(star):
    if star <= 2: return 'Negatif'
    elif star == 3: return 'Netral'
    else: return 'Positif'

df['sentiment'] = df['stars'].apply(convert_stars_to_sentiment)

# 2. Membagi data untuk Latih dan Uji (80% Latih, 20% Uji)
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['sentiment'], test_size=0.2, random_state=42)

# 3. Membuat Pipeline: Mengubah teks jadi angka (TF-IDF) lalu melatih ke SVM
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svm', SVC(kernel='linear', probability=True))
])

# 4. Memulai Proses Training (Model belajar pola bahasa di sini)
model_pipeline.fit(X_train, y_train)

# 1. Minta AI (model_pipeline) menebak jawaban untuk 20% data ujian (X_test)
print("\nMelakukan ujian pada data test...")
y_pred = model_pipeline.predict(X_test)

# 2. Tampilkan Rapor Hasil Evaluasinya (Classification Report)
print("\n=== RAPOR EVALUASI MODEL ===")
print(classification_report(y_test, y_pred))

# 3. Tampilkan Confusion Matrix (Tabel tebakan benar vs salah)
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

# 5. Ekspor Model menjadi File "Otak AI" (.pkl)
joblib.dump(model_pipeline, 'model_sentimen_muterbandung.pkl')
print("Model berhasil dilatih dan disimpan!")


Melakukan ujian pada data test...

=== RAPOR EVALUASI MODEL ===
              precision    recall  f1-score   support

     Negatif       0.82      0.78      0.80       447
      Netral       0.67      0.04      0.08       138
     Positif       0.91      0.98      0.94      1945

    accuracy                           0.89      2530
   macro avg       0.80      0.60      0.61      2530
weighted avg       0.88      0.89      0.87      2530


=== CONFUSION MATRIX ===
[[ 348    1   98]
 [  35    6   97]
 [  41    2 1902]]
Model berhasil dilatih dan disimpan!
